[Reference](https://medium.com/@sudharmamokashi18/from-messy-data-models-to-agentic-data-catalogs-building-an-ai-data-architect-with-okf-and-gemini-75a20caee9ed$0)

# Step 1: Treating Metadata as Code


## bronze_sales.md

```
---
type: BigQuery Table
title: Bronze Raw Sales
resource_id: bronze_sales.data_source_bnz
description: Raw, unvalidated point-of-sale transaction logs landed via batch ingestion.
tags: [bronze, raw, ingestion, pos]
timestamp: 2026-06-01T00:00:00Z
---

# Schema

| Column | Type | Description |
|---|---|---|
| `tx_id` | STRING | Raw transaction ID from POS terminal. |
| `raw_cust_ref` | STRING | Unstandardized customer string / card hash. |
| `raw_sku` | STRING | Barcode or product SKU. |
| `tx_amount` | NUMERIC | Total transaction cost. |
| `ingest_timestamp` | TIMESTAMP | Wall-clock time data landed in warehouse. |

# Notes & Known Quirks

* Contains duplicate records when terminals retry failed network requests.
* Downstream deduplication happens during transformation into [Gold Sales Facts](/gold/gold_sales_facts.md).
```

## Step 1: Treating Metadata as Code
```
---
type: BigQuery Table
title: Silver Customers
resource_id: silver_ds.d_cust
description: Cleansed, conformed customer dimension with deduplicated master records.
tags: [silver, dimension, customer]
timestamp: 2026-06-01T00:00:00Z
---

# Schema

| Column | Type | Description |
|---|---|---|
| `customer_id` | STRING | Primary Key. Standardized UUID. |
| `loyalty_tier` | STRING | Customer tier: 'Gold', 'Silver', 'Bronze', or 'Standard'. |
| `region` | STRING | Standardized 2-character country/region code. |

# Referenced by

- [Gold Sales Facts](/gold/gold_sales_facts.md) via `customer_id`
```

## silver_products.md
```
---
type: BigQuery Table
title: Silver Products
resource_id: silver_ds.d_prod
description: Conformed product dimension table mapped to standard category hierarchies.
tags: [silver, dimension, product, master-data]
timestamp: 2026-06-01T00:00:00Z
---

# Schema

| Column | Type | Description |
|---|---|---|
| `sku_id` | STRING | Primary Key. Standardized Global Trade Item Number (GTIN). |
| `category` | STRING | Product category (e.g., Beverages, Packaged Foods). |
| `unit_cost` | NUMERIC | Base production cost per unit in USD. |

# Referenced by

- [Gold Sales Facts](/gold/gold_sales_facts.md) via `sku_id`
- [Gold Inventory Summary](/gold/gold_inventory_summary.md) via `sku_id`
```

## gold_inventory_summary.md
```
---
type: BigQuery Table
title: Gold Inventory Summary
description: Daily snapshot of warehouse stock levels per product.
tags: [gold, aggregate, inventory, supply-chain]
timestamp: 2026-06-01T00:00:00Z
---

# Schema

| Column | Type | Description |
|---|---|---|
| `snapshot_date` | DATE | Date of the snapshot. |
| `sku_id` | STRING | Foreign Key to [Silver Products](/silver/silver_products.md). |
| `stock_on_hand` | INT64 | Total physical units in fulfillment centers. |

# Joins & Traversal

* Join with [Silver Products](/silver/silver_products.md) on `sku_id`.

# Notes

This is a separate lineage branch from [Gold Sales Facts](/gold/gold_sales_facts.md) —
both consume [Silver Products](/silver/silver_products.md) but track different metrics
(inventory levels vs. revenue).
```

# Step 2: Giving the AI a “Read” Tool


In [1]:
import os
from pathlib import Path

from google import genai
from google.genai import types


BUNDLE_ROOT = Path(__file__).parent / "lineage-bundle"


def read_okf_doc(path: str) -> str:
    """Reads a single OKF (Open Knowledge Format) concept document.

    Use this to fetch table schemas, metric definitions, or any concept
    referenced by a markdown link inside another OKF document (e.g. a
    link like '/gold/gold_sales_facts.md'). Every link in this bundle is
    root-relative — always pass the path exactly as it appears in the
    link, leading slash included. Follow every relevant markdown link in
    the returned content by calling this tool again until you have
    traced the full chain.

    Args:
        path: The root-relative OKF path to the concept doc, e.g.
            "/gold/gold_sales_facts.md" or "/bronze/bronze_sales.md".

    Returns:
        The full raw markdown content of the concept document (frontmatter
        + body), or an error message if the file does not exist.
    """
    relative = path.lstrip("/")
    target = (BUNDLE_ROOT / relative).resolve()

    if BUNDLE_ROOT not in target.parents and target != BUNDLE_ROOT:
        return f"Error: '{path}' resolves outside the OKF bundle."

    if not target.exists():
        return (
            f"Error: no concept doc found at '{path}' (resolved to {target}). "
            f"Check for filename typos or mismatched links."
        )

    return target.read_text(encoding="utf-8")


client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

config = types.GenerateContentConfig(
    system_instruction=(
        "You are an expert Data Platform Architect. You have direct access "
        "to our data catalog via the Open Knowledge Format. Whenever a user "
        "asks about SQL queries, schemas, or lineage, use the `read_okf_doc` "
        "tool to fetch concept documents. Always follow markdown hyperlinks "
        "inside a doc's body to traverse table relationships before "
        "answering, rather than guessing. If you don't know where to start, "
        "begin by looking at '/index.md' for the bundle overview, or "
        "'/gold/gold_sales_facts.md' and '/bronze/bronze_sales.md' directly."
    ),
    tools=[read_okf_doc],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=False),
    temperature=0.2,
)

def ask(prompt: str) -> None:
    print(f"\n{'=' * 70}\nQ: {prompt}\n{'=' * 70}")
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=config,
    )
    print(response.text)


if __name__ == "__main__":
    # --- 3. Example prompts for the blog / demo screenshots ------------
    ask(
        "Trace the full lineage from bronze_sales to gold_sales_facts. "
        "List every hop and what happens at each stage."
    )

# Step 3: The Agentic Loop


In [2]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

config = types.GenerateContentConfig(
    system_instruction=(
        "You are an expert Data Platform Architect. You have direct access "
        "to our data catalog via the Open Knowledge Format. Whenever a user "
        "asks about SQL queries, schemas, or lineage, use the `read_okf_doc` "
        "tool to fetch concept documents. Always follow markdown hyperlinks "
        "inside a doc's body to traverse table relationships..."
    ),
    tools=[read_okf_doc],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=False),
    temperature=0.2,
)